In [10]:
!pip install faiss-cpu sentence-transformers langchain transformers pypdf

In [12]:
!pip install PyPDF2

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 232.6/232.6 kB 14.7 MB/s eta 0:00:00


In [13]:
from PyPDF2 import PdfReader

pdf_path = "/content/story.pdf"   # 👈 Upload your PDF here
reader = PdfReader(pdf_path)

In [14]:
docs = []
for page in reader.pages:
    text = page.extract_text()
    if text:
        docs.append(text)

print(f"Loaded {len(docs)} pages from PDF")
print(docs[0][:500])

Loaded 1 pages from PDF
The Little Star and the Brave Boy  
Once upon a time, in a small village, there lived a boy named Arjun.  
Arjun loved to look at the night sky.  
Every night, he waved at the stars and whispered, “Goodnight, friends!”  
One evening, a tiny star twinkled brighter than the others.  
“Hello, Arjun!” the star whispered.  
Arjun rubbed his eyes. “Did you just talk?”  
“Yes!” said the star. “My name is Twink.”  
Twink was sad. “I have fallen from the sky. I don’t know how to go back.”  
Arjun smiled.


In [15]:
# 2. Split into Smaller Chunks (for better retrieval)
# -----------------------------------------------
from langchain.text_splitter import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=100)
chunks = text_splitter.split_text(" ".join(docs))

print(f"Total Chunks Created: {len(chunks)}")
print(chunks[0])

Total Chunks Created: 4
The Little Star and the Brave Boy  
Once upon a time, in a small village, there lived a boy named Arjun.  
Arjun loved to look at the night sky.  
Every night, he waved at the stars and whispered, “Goodnight, friends!”  
One evening, a tiny star twinkled brighter than the others.  
“Hello, Arjun!” the star whispered.  
Arjun rubbed his eyes. “Did you just talk?”  
“Yes!” said the star. “My name is Twink.”  
Twink was sad. “I have fallen from the sky. I don’t know how to go back.”


In [16]:
# 3. Generate Embeddings
# -----------------------------------------------
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer("all-MiniLM-L6-v2")  # Fast + good quality
doc_embeddings = embedding_model.encode(chunks, convert_to_tensor=True)

print("Embeddings shape:", doc_embeddings.shape)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embeddings shape: torch.Size([4, 384])


In [17]:
import faiss
import numpy as np

doc_embeddings_np = doc_embeddings.cpu().detach().numpy()
dimension = doc_embeddings_np.shape[1]

index = faiss.IndexFlatL2(dimension)
index.add(doc_embeddings_np)

print("Number of vectors in index:", index.ntotal)

Number of vectors in index: 4


In [18]:
# 5. RAG Query Function
# -----------------------------------------------
from transformers import pipeline

# Load HuggingFace LLM (you can replace with OpenAI if you have API key)
qa_pipeline = pipeline("text2text-generation", model="google/flan-t5-base")

def rag_query(question, k=3):
    # Embed query
    query_embedding = embedding_model.encode([question])

    # Search FAISS
    distances, indices = index.search(np.array(query_embedding).astype("float32"), k)
    retrieved_docs = [chunks[i] for i in indices[0]]

    # Create context
    context = " ".join(retrieved_docs)

    # Generate answer
    prompt = f"Answer the question based on the context below:\n\nContext: {context}\n\nQuestion: {question}\nAnswer:"
    answer = qa_pipeline(prompt, max_length=200, truncation=True)[0]['generated_text']

    return answer, retrieved_docs

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/990M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

Device set to use cpu


In [ ]:
# -----------------------------------------------
# 6. Test with 5 Questions
# -----------------------------------------------
questions = [
    "What was the boy’s name in the story?",
    "Who was Twink?",
    "What did Arjun pack in his backpack to help Twink?",
    "How did Twink finally reach the sky again?",
    "What did Twink do at night to show his friendship to Arjun?"
]

for q in questions:
    answer, retrieved = rag_query(q)
    print(f"\n🔹 Question: {q}")
    print(f"✅ Answer: {answer}")
    print(f"📚 Retrieved Docs Snippet: {[d[:120] for d in retrieved]}")


Both `max_new_tokens` (=256) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



🔹 Question: What was the boy’s name in the story?
✅ Answer: Arjun
📚 Retrieved Docs Snippet: ['The Little Star and the Brave Boy  \nOnce upon a time, in a small village, there lived a boy named Arjun.  \nArjun loved t', 'That night, Arjun looked up.  \nTwink sparkled extra bright, just for him.  \nWhenever Arjun felt lonely, Twink winked dow', 'Twink was sad. “I have fallen from the sky. I don’t know how to go back.”  \nArjun smiled. “Don’t worry, Twink. I’ll help']


Both `max_new_tokens` (=256) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
